[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ImagingDataCommons/CloudSegmentator/blob/main/workflows/MOOSE/Notebooks/moosePostProcessNotebook.ipynb)

# **MOOSE Post-Processing (Task 2): Convert NIfTI segmentations to DICOM-SEG**

This notebook is the CPU-side of the MOOSE twoVM workflow. It:
1. Extracts the `moose_segmentations.tar.lz4` archive produced by Task 1
2. Re-downloads source DICOM from IDC for each series (needed by dcmqi as reference)
3. For each series/model pair, builds a dcmqi config JSON and runs `itkimage2segimage`
4. Tars + lz4-compresses all DICOM-SEG files as `moose_dicom_seg.tar.lz4`

Please cite:

Herz C, Fillion-Robin JC, Onken M, Riesmeier J, Lasso A, Pinter C, Fichtinger G, Pieper S, Clunie D, Kikinis R, Fedorov A. dcmqi: An Open Source Library for Standardized Communication of Quantitative Image Analysis Results Using DICOM. Cancer Res. 2017 Nov 1;77(21):e87-e90. https://doi.org/10.1158/0008-5472.CAN-17-0336

Shiyam Sundar LK, et al. Fully automated, semantic segmentation of whole-body 18F-FDG PET/CT images based on data-centric artificial intelligence. J Nucl Med. 2022. https://doi.org/10.2967/jnumed.122.264063

## Imports

In [ ]:
import json
import os
import re
import shutil
import subprocess
import time
import traceback
from pathlib import Path

import nibabel as nib
import numpy as np
from idc_index.index import IDCClient


## Parameters

Tagged `parameters` so papermill can override via `-p segmentationArchivePath <path>`.

In [ ]:
# Path to the lz4-compressed tar produced by Task 1 (mooseInferenceNotebook)
segmentationArchivePath = "moose_segmentations.tar.lz4"

## MOOSE organ label tables

moosez emits multi-label NIfTI segmentations where integer labels correspond to anatomical structures. The dictionaries below mirror `moosez.constants.ORGAN_INDICES` for the clinical CT models supported by this workflow. Any label encountered at runtime that is not in the dictionary will be emitted as a generic `segment_<id>` with a placeholder SNOMED code.

In [ ]:
ORGAN_INDICES = {
    "clin_ct_organs": {
        1: "adrenal_gland_left", 2: "adrenal_gland_right", 3: "bladder",
        4: "brain", 5: "gallbladder", 6: "kidney_left", 7: "kidney_right",
        8: "liver", 9: "lung_lower_lobe_left", 10: "lung_lower_lobe_right",
        11: "lung_middle_lobe_right", 12: "lung_upper_lobe_left",
        13: "lung_upper_lobe_right", 14: "pancreas", 15: "spleen",
        16: "stomach", 17: "thyroid_left", 18: "thyroid_right", 19: "trachea",
    },
    "clin_ct_lungs": {
        1: "lung_upper_lobe_left", 2: "lung_lower_lobe_left",
        3: "lung_upper_lobe_right", 4: "lung_middle_lobe_right",
        5: "lung_lower_lobe_right",
    },
    "clin_ct_ribs": {
        i: (f"rib_left_{i}" if i <= 12 else f"rib_right_{i - 12}")
        for i in range(1, 25)
    },
    "clin_ct_vertebrae": {
        1: "vertebra_L5", 2: "vertebra_L4", 3: "vertebra_L3",
        4: "vertebra_L2", 5: "vertebra_L1", 6: "vertebra_T12",
        7: "vertebra_T11", 8: "vertebra_T10", 9: "vertebra_T9",
        10: "vertebra_T8", 11: "vertebra_T7", 12: "vertebra_T6",
        13: "vertebra_T5", 14: "vertebra_T4", 15: "vertebra_T3",
        16: "vertebra_T2", 17: "vertebra_T1", 18: "vertebra_C7",
        19: "vertebra_C6", 20: "vertebra_C5", 21: "vertebra_C4",
        22: "vertebra_C3", 23: "vertebra_C2", 24: "vertebra_C1",
    },
    "clin_ct_body": {1: "legs", 2: "body", 3: "arms"},
    "clin_ct_cardiac": {
        1: "heart_myocardium", 2: "heart_atrium_left", 3: "heart_atrium_right",
        4: "heart_ventricle_left", 5: "heart_ventricle_right",
        6: "aorta", 7: "pulmonary_artery", 8: "iliac_artery_left",
        9: "iliac_artery_right", 10: "iliac_vena_left", 11: "iliac_vena_right",
        12: "inferior_vena_cava", 13: "portal_splenic_vein",
    },
    "clin_ct_muscles": {
        1: "autochthon_left", 2: "autochthon_right",
        3: "iliopsoas_left", 4: "iliopsoas_right",
        5: "gluteus_maximus_left", 6: "gluteus_maximus_right",
        7: "gluteus_medius_left", 8: "gluteus_medius_right",
        9: "gluteus_minimus_left", 10: "gluteus_minimus_right",
    },
}


def organ_name(model: str, label_id: int) -> str:
    return ORGAN_INDICES.get(model, {}).get(label_id, f"segment_{label_id}")


# Regex to extract the moosez model identifier from segmentation filenames.
# moosez's Python API writes: {multilabel_prefix}segmentation_{file_stem}.nii.gz
# where multilabel_prefix = "{imaging_type}_{MODALITY}_{region}_"
# e.g. "clin_CT_organs_segmentation_CT_foo.nii.gz" -> model key "clin_ct_organs"
_MOOSE_SEG_FILENAME_RE = re.compile(
    r"^((?:clin|preclin)_[A-Z0-9_-]+?_[a-z][a-z0-9_]*)_segmentation_"
)


def model_from_filename(filename: str) -> str:
    """Return the ORGAN_INDICES key encoded in a moosez segmentation filename.

    The multilabel_prefix uses an uppercase modality token
    (e.g. 'clin_CT_organs_'), while ORGAN_INDICES keys are fully
    lowercase (e.g. 'clin_ct_organs').
    Returns an empty string when the filename pattern is not recognised.
    """
    m = _MOOSE_SEG_FILENAME_RE.match(filename)
    return m.group(1).lower() if m else ""


def segment_color(label_id: int) -> list:
    """Return a visually distinct [R, G, B] display colour for a label integer."""
    _PALETTE = [
        [255,  85,   0], [  0, 128, 255], [  0, 200,  80], [255, 218,   0],
        [220,   0, 220], [  0, 220, 220], [255, 128,   0], [100, 100, 255],
        [200,   0, 100], [  0, 180, 140], [128, 255,   0], [255,   0, 128],
        [  0, 255, 180], [160,  80, 255], [255, 200, 100], [ 80, 220,  80],
        [255,  50, 150], [ 50, 200, 255], [180, 255,  50], [100, 200, 200],
    ]
    return _PALETTE[(label_id - 1) % len(_PALETTE)]


# Inlined from https://github.com/ENHANCE-PET/MOOSE/blob/d844de3424662a6ce35d531c9380c1c55be44113/moosez/mappings/SNOMED.py
# moosez is not installed on the post-process VM, so we copy the dict here directly.
moose_to_snomed = {
    # cardiac / vascular
    "heart_myocardium":      {"ID": "74281007",   "name": "Myocardium structure"},
    "heart_atrium_left":     {"ID": "82471001",   "name": "Left atrial structure"},
    "heart_atrium_right":    {"ID": "73829009",   "name": "Right atrial structure"},
    "heart_ventricle_left":  {"ID": "87878005",   "name": "Left ventricular structure"},
    "heart_ventricle_right": {"ID": "53085002",   "name": "Right ventricular structure"},
    "aorta":                 {"ID": "15825003",   "name": "Aortic structure"},
    "iliac_artery_left":     {"ID": "721077009",  "name": "Structure of left iliac artery"},
    "iliac_artery_right":    {"ID": "721035009",  "name": "Structure of right iliac artery"},
    "iliac_vena_left":       {"ID": "244411005",  "name": "Iliac vein structure"},
    "iliac_vena_right":      {"ID": "244411005",  "name": "Iliac vein structure"},
    "inferior_vena_cava":    {"ID": "64131007",   "name": "Inferior vena cava structure"},
    "portal_splenic_vein":   {"ID": "35819009",   "name": "Structure of splenic vein"},
    "pulmonary_artery":      {"ID": "81040000",   "name": "Pulmonary artery structure"},
    # gastrointestinal
    "colon":                 {"ID": "71854001",   "name": "Colon structure"},
    "duodenum":              {"ID": "38848004",   "name": "Duodenum structure"},
    "esophagus":             {"ID": "32849002",   "name": "Esophageal structure"},
    "small_bowel":           {"ID": "30315005",   "name": "Small intestinal structure"},
    # muscles
    "autochthon_left":       {"ID": "44947003",   "name": "Structure of erector spinae muscle"},
    "autochthon_right":      {"ID": "44947003",   "name": "Structure of erector spinae muscle"},
    "gluteus_maximus_left":  {"ID": "206007",     "name": "Structure of gluteus maximus muscle"},
    "gluteus_maximus_right": {"ID": "206007",     "name": "Structure of gluteus maximus muscle"},
    "gluteus_medius_left":   {"ID": "78333006",   "name": "Structure of gluteus medius muscle"},
    "gluteus_medius_right":  {"ID": "78333006",   "name": "Structure of gluteus medius muscle"},
    "gluteus_minimus_left":  {"ID": "75297007",   "name": "Structure of gluteus minimus muscle"},
    "gluteus_minimus_right": {"ID": "75297007",   "name": "Structure of gluteus minimus muscle"},
    "iliopsoas_left":        {"ID": "68455001",   "name": "Structure of iliopsoas muscle"},
    "iliopsoas_right":       {"ID": "68455001",   "name": "Structure of iliopsoas muscle"},
    # organs
    "adrenal_gland_left":    {"ID": "12003004",   "name": "Left adrenal gland"},
    "adrenal_gland_right":   {"ID": "29392005",   "name": "Right adrenal gland"},
    "bladder":               {"ID": "89837001",   "name": "Urinary bladder structure"},
    "brain":                 {"ID": "12738006",   "name": "Brain structure"},
    "gallbladder":           {"ID": "28231008",   "name": "Gallbladder structure"},
    "kidney_left":           {"ID": "18639004",   "name": "Left kidney structure"},
    "kidney_right":          {"ID": "9846003",    "name": "Right kidney structure"},
    "liver":                 {"ID": "10200004",   "name": "Liver structure"},
    "lung_lower_lobe_left":  {"ID": "41224006",   "name": "Lower lobe of left lung"},
    "lung_lower_lobe_right": {"ID": "266005",     "name": "Lower lobe of right lung"},
    "lung_middle_lobe_right":{"ID": "72481006",   "name": "Middle lobe of right lung"},
    "lung_upper_lobe_left":  {"ID": "44714003",   "name": "Upper lobe of left lung"},
    "lung_upper_lobe_right": {"ID": "42400003",   "name": "Upper lobe of right lung"},
    "pancreas":              {"ID": "15776009",   "name": "Pancreatic structure"},
    "spleen":                {"ID": "78961009",   "name": "Splenic structure"},
    "stomach":               {"ID": "69695003",   "name": "Stomach structure"},
    "thyroid_left":          {"ID": "79163004",   "name": "Left lobe of thyroid gland"},
    "thyroid_right":         {"ID": "29565003",   "name": "Right lobe of thyroid gland"},
    "trachea":               {"ID": "44567001",   "name": "Trachea structure"},
    # skeletal — appendicular
    "carpal_left":           {"ID": "719623009",  "name": "Bone structure of left carpus"},
    "carpal_right":          {"ID": "719624003",  "name": "Bone structure of right carpus"},
    "clavicle_left":         {"ID": "720617006",  "name": "Bone structure of left clavicle"},
    "clavicle_right":        {"ID": "720616002",  "name": "Bone structure of right clavicle"},
    "femur_left":            {"ID": "1356755002", "name": "Entire bone of left femur"},
    "femur_right":           {"ID": "1356756001", "name": "Entire bone of right femur"},
    "fibula_left":           {"ID": "1017225004", "name": "Entire left fibula"},
    "fibula_right":          {"ID": "1017224000", "name": "Entire right fibula"},
    "fingers_left":          {"ID": "762870007",  "name": "Bone structure of phalanx of left hand"},
    "fingers_right":         {"ID": "762871006",  "name": "Bone structure of phalanx of right hand"},
    "humerus_left":          {"ID": "719460003",  "name": "Bone structure of left humerus"},
    "humerus_right":         {"ID": "719461004",  "name": "Bone structure of right humerus"},
    "metacarpal_left":       {"ID": "762008002",  "name": "Structure of metacarpal bone of left hand"},
    "metacarpal_right":      {"ID": "762009005",  "name": "Structure of metacarpal bone of right hand"},
    "metatarsal_left":       {"ID": "726438004",  "name": "Structure of metatarsal bone of left foot"},
    "metatarsal_right":      {"ID": "726439007",  "name": "Structure of metatarsal bone of right foot"},
    "patella_left":          {"ID": "734208001",  "name": "Bone structure of left patella"},
    "patella_right":         {"ID": "734210004",  "name": "Bone structure of right patella"},
    "radius_left":           {"ID": "719464007",  "name": "Bone structure of left radius"},
    "radius_right":          {"ID": "719465008",  "name": "Bone structure of right radius"},
    "scapula_left":          {"ID": "719627005",  "name": "Bone structure of left scapula"},
    "scapula_right":         {"ID": "719628000",  "name": "Bone structure of right scapula"},
    "skull":                 {"ID": "118646007",  "name": "Entire bone of head"},
    "sternum":               {"ID": "302522007",  "name": "Entire sternum"},
    "tarsal_left":           {"ID": "108371006",  "name": "Bone structure of tarsus"},
    "tarsal_right":          {"ID": "108371006",  "name": "Bone structure of tarsus"},
    "tibia_left":            {"ID": "719492002",  "name": "Bone structure of left tibia"},
    "tibia_right":           {"ID": "719491009",  "name": "Bone structure of right tibia"},
    "toes_left":             {"ID": "720642002",  "name": "Bone structure of phalanx of left foot"},
    "toes_right":            {"ID": "720641009",  "name": "Bone structure of phalanx of right foot"},
    "ulna_left":             {"ID": "719462006",  "name": "Bone structure of left ulna"},
    "ulna_right":            {"ID": "719463001",  "name": "Bone structure of right ulna"},
    # ribs
    "rib_left_1":            {"ID": "1354709002", "name": "Bone structure of left first rib"},
    "rib_left_2":            {"ID": "1354712004", "name": "Bone structure of left second rib"},
    "rib_left_3":            {"ID": "1354715002", "name": "Bone structure of left third rib"},
    "rib_left_4":            {"ID": "1354718000", "name": "Bone structure of left fourth rib"},
    "rib_left_5":            {"ID": "1354724006", "name": "Bone structure of left fifth rib"},
    "rib_left_6":            {"ID": "1354726008", "name": "Bone structure of left sixth rib"},
    "rib_left_7":            {"ID": "1354728009", "name": "Bone structure of left seventh rib"},
    "rib_left_8":            {"ID": "1354733008", "name": "Bone structure of left eighth rib"},
    "rib_left_9":            {"ID": "1354735001", "name": "Bone structure of left ninth rib"},
    "rib_left_10":           {"ID": "1354737009", "name": "Bone structure of left tenth rib"},
    "rib_left_11":           {"ID": "1354742001", "name": "Bone structure of left eleventh rib"},
    "rib_left_12":           {"ID": "1354744000", "name": "Bone structure of left twelfth rib"},
    "rib_left_13":           {"ID": "205460009",  "name": "Accessory rib"},
    "rib_right_1":           {"ID": "1354710007", "name": "Bone structure of right first rib"},
    "rib_right_2":           {"ID": "1354713009", "name": "Bone structure of right second rib"},
    "rib_right_3":           {"ID": "1354716001", "name": "Bone structure of right third rib"},
    "rib_right_4":           {"ID": "1354719008", "name": "Bone structure of right fourth rib"},
    "rib_right_5":           {"ID": "1354725007", "name": "Bone structure of right fifth rib"},
    "rib_right_6":           {"ID": "1354727004", "name": "Bone structure of right sixth rib"},
    "rib_right_7":           {"ID": "1354729001", "name": "Bone structure of right seventh rib"},
    "rib_right_8":           {"ID": "1354734002", "name": "Bone structure of right eighth rib"},
    "rib_right_9":           {"ID": "1354736000", "name": "Bone structure of right ninth rib"},
    "rib_right_10":          {"ID": "1354738004", "name": "Bone structure of right tenth rib"},
    "rib_right_11":          {"ID": "1354743006", "name": "Bone structure of right eleventh rib"},
    "rib_right_12":          {"ID": "1354745004", "name": "Bone structure of right twelfth rib"},
    "rib_right_13":          {"ID": "205460009",  "name": "Accessory rib"},
    # vertebrae
    "vertebra_C1":           {"ID": "731658004",  "name": "Entire bone of atlas"},
    "vertebra_C2":           {"ID": "731673003",  "name": "Entire bone of axis"},
    "vertebra_C3":           {"ID": "731721006",  "name": "Entire bone of C3"},
    "vertebra_C4":           {"ID": "731650006",  "name": "Entire bone of C4"},
    "vertebra_C5":           {"ID": "731670000",  "name": "Entire bone of C5"},
    "vertebra_C6":           {"ID": "731668009",  "name": "Entire bone of C6"},
    "vertebra_C7":           {"ID": "731715007",  "name": "Entire bone of C7"},
    "vertebra_T1":           {"ID": "731692001",  "name": "Entire bone of T1"},
    "vertebra_T2":           {"ID": "731683004",  "name": "Entire bone of T2"},
    "vertebra_T3":           {"ID": "731648003",  "name": "Entire bone of T3"},
    "vertebra_T4":           {"ID": "731700006",  "name": "Entire bone of T4"},
    "vertebra_T5":           {"ID": "731685006",  "name": "Entire bone of T5"},
    "vertebra_T6":           {"ID": "731676006",  "name": "Entire bone of T6"},
    "vertebra_T7":           {"ID": "731689000",  "name": "Entire bone of T7"},
    "vertebra_T8":           {"ID": "731654002",  "name": "Entire bone of T8"},
    "vertebra_T9":           {"ID": "731711003",  "name": "Entire bone of T9"},
    "vertebra_T10":          {"ID": "731652003",  "name": "Entire bone of T10"},
    "vertebra_T11":          {"ID": "731656000",  "name": "Entire bone of T11"},
    "vertebra_T12":          {"ID": "731662005",  "name": "Entire bone of T12"},
    "vertebra_L1":           {"ID": "731695004",  "name": "Entire bone of L1"},
    "vertebra_L2":           {"ID": "731657009",  "name": "Entire bone of L2"},
    "vertebra_L3":           {"ID": "731669001",  "name": "Entire bone of L3"},
    "vertebra_L4":           {"ID": "731655001",  "name": "Entire bone of L4"},
    "vertebra_L5":           {"ID": "731680001",  "name": "Entire bone of L5"},
    "vertebra_L6":           {"ID": "82137000",   "name": "Entire bone of L6"},
    # hip / pelvis
    "hip_left":              {"ID": "287679003",  "name": "Left hip region structure"},
    "hip_right":             {"ID": "287579007",  "name": "Right hip region structure"},
    "sacrum":                {"ID": "699698002",  "name": "Sacrum"},
    # body composition
    "skeletal_muscle":       {"ID": "127954009",  "name": "Skeletal muscle structure"},
    "subcutaneous_fat":      {"ID": "67769002",   "name": "Subcutaneous adipose tissue"},
    "visceral_fat":          {"ID": "725273006",  "name": "Structure of adipose tissue of abdomen"},
}


## Extract the segmentation archive

In [ ]:
EXTRACT_DIR = Path("/tmp/moose_extract")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

subprocess.run(
    f"lz4 -d -c {segmentationArchivePath} | tar -xf - -C {EXTRACT_DIR}",
    shell=True,
    check=True,
)

top_dirs = [p for p in EXTRACT_DIR.iterdir() if p.is_dir()]
if len(top_dirs) == 1:
    MOOSE_ROOT = top_dirs[0]
else:
    MOOSE_ROOT = EXTRACT_DIR

series_dirs = sorted([p for p in MOOSE_ROOT.iterdir() if p.is_dir()])
print(f"Found {len(series_dirs)} series in {MOOSE_ROOT}")
for p in series_dirs:
    print(f"  {p.name}")

## Helpers

In [ ]:
idc_client = IDCClient()

# Regex matches moosez per-model output subdirectories: moosez-<model>-<timestamp>
MODEL_DIR_RE = re.compile(r"^moosez-(?P<model>.+?)-\d{4}[-_]?\d{2}[-_]?\d{2}[-_T]?\d{2}[-_:]?\d{2}[-_:]?\d{2}$")


def download_dicom(uid: str, dest: Path) -> None:
    """Download a DICOM series from IDC into dest."""
    dest.mkdir(parents=True, exist_ok=True)
    idc_client.download_from_selection(
        downloadDir=str(dest),
        seriesInstanceUID=uid,
    )


def find_series_number(dicom_dir: Path) -> str:
    """Peek at one DICOM file to get the SeriesNumber for metadata."""
    try:
        import pydicom
        for p in dicom_dir.rglob("*.dcm"):
            ds = pydicom.dcmread(str(p), stop_before_pixels=True)
            sn = getattr(ds, "SeriesNumber", None)
            if sn is not None:
                return str(int(sn))
    except Exception:
        pass
    return "1"


def build_dcmqi_config(model: str, labels: list, series_number: str) -> dict:
    """Build a dcmqi itkimage2segimage metadata JSON for a single multi-label mask."""
    _GENERIC_CATEGORY = {
        "CodeValue": "123037004",
        "CodingSchemeDesignator": "SCT",
        "CodeMeaning": "Anatomical Structure",
    }
    _GENERIC_TYPE = {
        "CodeValue": "246183007",
        "CodingSchemeDesignator": "SCT",
        "CodeMeaning": "Body structure",
    }

    segments = []
    for label_id in labels:
        name = organ_name(model, int(label_id))
        snomed_entry = moose_to_snomed.get(name)
        display_name = snomed_entry["name"] if snomed_entry else name
        seg = {
            "labelID": int(label_id),
            "SegmentDescription": display_name,
            "SegmentLabel": display_name,
            "SegmentAlgorithmType": "AUTOMATIC",
            "SegmentAlgorithmName": "moosez",
            "SegmentedPropertyCategoryCodeSequence": _GENERIC_CATEGORY,
            "SegmentedPropertyTypeCodeSequence": {
                "CodeValue": snomed_entry["ID"],
                "CodingSchemeDesignator": "SCT",
                "CodeMeaning": snomed_entry["name"],
            } if snomed_entry else _GENERIC_TYPE,
            "recommendedDisplayRGBValue": segment_color(label_id),
        }
        segments.append(seg)

    return {
        "ContentCreatorName": "MOOSE^CloudSegmentator",
        "ClinicalTrialSeriesID": "Session1",
        "ClinicalTrialTimePointID": "1",
        "SeriesDescription": f"MOOSE ({model}) Segmentation",
        "SeriesNumber": str(int(series_number) * 100 + 1) if series_number.isdigit() else "100",
        "InstanceNumber": "1",
        "BodyPartExamined": "",
        "segmentationType": "LABELMAP",
        "segmentAttributes": [segments],
        "ContentLabel": "SEGMENTATION",
        "ContentDescription": f"moosez {model} multi-label segmentation",
        "ClinicalTrialCoordinatingCenterName": "",
    }


def unique_nonzero_labels(nifti_path: Path) -> list:
    arr = np.asanyarray(nib.load(str(nifti_path)).dataobj)
    return sorted(int(v) for v in np.unique(arr) if v != 0)


## Generate DICOM-SEG per series/model

In [ ]:
DICOM_DIR = Path("/tmp/dicom")
DICOM_SEG_DIR = Path("/tmp/moose_dicom_seg")
CONFIG_DIR = Path("/tmp/moose_configs")
for d in (DICOM_DIR, DICOM_SEG_DIR, CONFIG_DIR):
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True)

dicom_seg_errors = []
usage_metrics = {"series": {}}

for series_dir in series_dirs:
    uid = series_dir.name
    print(f"\n=== Processing {uid} ===")
    series_start = time.time()

    # Re-download source DICOM (dcmqi needs it as the image reference)
    dicom_dest = DICOM_DIR / uid
    try:
        t0 = time.time()
        download_dicom(uid, dicom_dest)
        usage_metrics["series"].setdefault(uid, {})["download_s"] = round(time.time() - t0, 1)
    except Exception as exc:
        msg = f"{uid}: download failed: {exc}\n{traceback.format_exc()}"
        dicom_seg_errors.append(msg)
        print(f"ERROR: {msg}")
        continue

    series_number = find_series_number(dicom_dest)

    seg_out_dir = DICOM_SEG_DIR / uid
    seg_out_dir.mkdir(parents=True, exist_ok=True)

    # Prefer model subdirectories. If absent, treat this series as a flat legacy layout.
    model_dirs = [p for p in series_dir.iterdir() if p.is_dir()]
    synthetic_model_dirs = False
    if not model_dirs:
        flat_seg_files = sorted(series_dir.rglob("*.nii.gz"))
        if flat_seg_files:
            synthetic_model_dirs = True
            model_dirs = [series_dir]
            print(f"INFO: {uid}: using legacy flat segmentation layout ({len(flat_seg_files)} files)")
        else:
            msg = f"{uid}: no model directories or .nii.gz segmentations found in {series_dir}"
            dicom_seg_errors.append(msg)
            print(f"WARN: {msg}")
            continue

    for model_dir in model_dirs:
        if synthetic_model_dirs:
            model = "legacy"
            seg_files = sorted(model_dir.rglob("*.nii.gz"))
        else:
            m = MODEL_DIR_RE.match(model_dir.name)
            model = m.group("model") if m else model_dir.name

            # moosez places outputs under .../segmentations/*.nii.gz
            seg_files = list((model_dir / "segmentations").glob("*.nii.gz")) \
                if (model_dir / "segmentations").exists() \
                else list(model_dir.rglob("*.nii.gz"))

        if not seg_files:
            msg = f"{uid}/{model}: no .nii.gz segmentations under {model_dir}"
            dicom_seg_errors.append(msg)
            print(f"WARN: {msg}")
            continue

        for seg_idx, seg_file in enumerate(seg_files):
            try:
                labels = unique_nonzero_labels(seg_file)
                if not labels:
                    print(f"  {model}/{seg_file.name}: empty mask, skipping")
                    continue

                # If the directory-derived model name is unknown to ORGAN_INDICES,
                # try to retrieve it from the segmentation filename. moosez encodes
                # the model in its multilabel_prefix:
                #   "clin_CT_organs_segmentation_*.nii.gz" -> "clin_ct_organs"
                effective_model = model
                if effective_model not in ORGAN_INDICES:
                    fn_model = model_from_filename(seg_file.name)
                    if fn_model:
                        effective_model = fn_model

                cfg = build_dcmqi_config(effective_model, labels, series_number)
                cfg_path = CONFIG_DIR / f"{uid}_{effective_model}_{seg_idx}.json"
                cfg_path.write_text(json.dumps(cfg, indent=2))

                out_dcm = seg_out_dir / f"{effective_model}_{seg_idx}.dcm"
                t0 = time.time()
                result = subprocess.run(
                    [
                        "itkimage2segimage",
                        "--inputImageList", str(seg_file),
                        "--inputDICOMDirectory", str(dicom_dest),
                        "--outputDICOM", str(out_dcm),
                        "--inputMetadata", str(cfg_path),
                        "--skip",
                    ],
                    capture_output=True,
                    text=True,
                )
                elapsed = round(time.time() - t0, 1)

                if result.returncode != 0 or not out_dcm.exists():
                    msg = (
                        f"{uid}/{model}/{seg_file.name}: itkimage2segimage failed "
                        f"(rc={result.returncode})\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}"
                    )
                    dicom_seg_errors.append(msg)
                    print(f"  ERROR: {msg}")
                else:
                    usage_metrics["series"].setdefault(uid, {}).setdefault("models", {})[effective_model] = elapsed
                    print(f"  {effective_model}: wrote {out_dcm.name} ({elapsed}s, {len(labels)} segments)")

            except Exception as exc:
                msg = f"{uid}/{model}/{seg_file.name}: {traceback.format_exc()}"
                dicom_seg_errors.append(msg)
                print(f"  ERROR: {exc}")

    usage_metrics["series"].setdefault(uid, {})["total_s"] = round(time.time() - series_start, 1)

    # Clean up DICOM for this series to conserve disk
    shutil.rmtree(dicom_dest, ignore_errors=True)

if dicom_seg_errors:
    Path("dicom_seg_error_file.txt").write_text("\n\n".join(dicom_seg_errors))


## Package DICOM-SEG outputs

In [ ]:
def compress_dir(src_dir: Path, out_file: str) -> None:
    cmd = f"tar -cf - -C {src_dir.parent} {src_dir.name} | lz4 > {out_file}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Compression failed: {result.stderr}")
    size_mb = Path(out_file).stat().st_size / (1024 ** 2)
    print(f"Created {out_file} ({size_mb:.1f} MB)")

compress_dir(DICOM_SEG_DIR, "moose_dicom_seg.tar.lz4")

## Write usage metrics

In [ ]:
metrics_json = json.dumps(usage_metrics, indent=2)
metrics_path = Path("moose_postprocess_UsageMetrics.json")
metrics_path.write_text(metrics_json)

subprocess.run(
    f"lz4 -f {metrics_path} moose_postprocess_UsageMetrics.lz4",
    shell=True,
    check=True,
)

print(metrics_json)

## Summary

In [ ]:
print("=" * 60)
print("MOOSE Post-Process Summary")
print("=" * 60)
print(f"  Series processed : {len(series_dirs)}")
print(f"  Errors logged    : {len(dicom_seg_errors)}")
print("=" * 60)

if len(dicom_seg_errors) and len(dicom_seg_errors) >= len(series_dirs):
    raise RuntimeError("DICOM-SEG generation failed for all series - see dicom_seg_error_file.txt")